# 02 — RBI sentiment events

Requires `python scripts/run_batch_nlp_pipeline.py` after PDFs sit in `data/raw/mpc_archive/`.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

import pandas as pd
from config import DUCKDB_PATH, PROCESSED_DATA_PATH, MPC_ARCHIVE_PATH
from data_pipeline.db_utils import DatabaseManager

print("Archive PDFs:", sorted(p.name for p in Path(MPC_ARCHIVE_PATH).glob("*.pdf")))

def load(name):
    pq = PROCESSED_DATA_PATH / f"{name}.parquet"
    if pq.exists():
        return pd.read_parquet(pq)
    db = DatabaseManager(DUCKDB_PATH)
    if db.table_exists(name):
        return db.load_table(name).to_pandas()
    return pd.DataFrame()

ev = load("sentiment_events")
if ev.empty:
    print("No sentiment_events yet.")
else:
    ev["date"] = pd.to_datetime(ev["date"])
    display_cols = [c for c in ev.columns if c in ("date", "sentiment_score", "source", "reasoning") or True]
    print(ev.sort_values("date").tail(10))


In [ ]:
reg = load("regime_predictions")
if ev.empty or reg.empty:
    print("Need both sentiment_events and regime_predictions.")
else:
    import matplotlib.pyplot as plt
    reg["date"] = pd.to_datetime(reg["date"])
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(reg["date"], reg["macro_sentiment"], color="#276749", lw=1.2)
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_title("Forward-filled RBI stance")
    ax.set_ylabel("sentiment")
    plt.tight_layout()
    plt.show()
